# Decorator

## TOC

- [Intro](#intro)
- [Decorators with parameters](#decorators-with-parameters)
- [Data in decorators](#data-in-decorators)
- [Decorating methods](#decorating-methods)
- [Classes as decorators](#classes-as-decorators)
- [Decorating classes](#decorating-classes)
- [Wrapping coroutines](#wrapping-coroutines)
- [Docstrings of decorated functions](#docstrings-of-decorated-functions)
- [Wrapping up](#wrapping-up)
- [Summary](#summary)

### Intro

Suppose there is a *foo* [function](https://docs.python.org/3/glossary.html#term-function) that accepts a string and returns its modified version. Imagine that this function is crucial and neither its signature nor body can be changed. In certain cases, the default formatted message must be different, so the task is "to enclose the original message in brackets". Hm, easy-peasy, yet another function that reuses the target (*foo*) function.

In [1]:
def foo(s: str) -> str:
    """Returns a fancy string.

    Args:
        s (str): original string

    Returns:
        str
    """
    return f"string is {s!r}"


def add_brackets_v1(s: str) -> str:
    return f"[{foo(s)}]"


s = 'hello'
print(f"original: {foo(s)}")
print(f"modified: {add_brackets_v1(s)}")

original: string is 'hello'
modified: [string is 'hello']


An ugly solution for `add_brackets` depends on *foo*, so it is just a function that can reuse only the *foo* function and that is all. Ok, my bad, `add_brackets` can be less coupled with *foo* and be used with any string.

In [2]:
def add_brackets_v2(s: str) -> str:
    return f"[{s}]"


s = 'hello'
print(f"original: {foo(s)}")
print(f"modified: {add_brackets_v2(foo(s))}")

original: string is 'hello'
modified: [string is 'hello']


Later, the new wish combinations are asked, so now to maintain four cases:
1. a string in brackets (original -> "[s]")
2. a string in parentheses ("(s)")
3. case 1 and then case 2 ("([s])")
4. case 2 and then case 1 ("[(s)]")

Damn, revise the `add_brackets` and define `add_parentheses` and defi...no, their composition is enough (nested calls)!

In [3]:
def add_brackets(s: str) -> str:
    return f"[{s}]"


def add_parentheses(s: str) -> str:
    return f"({s})"


s = 'hello'
print(f"foo(s): {foo(s)}")
print(f"[foo(s)]: {add_brackets(foo(s))}")
print(f"(foo(s)): {add_parentheses(foo(s))}")
print(f"[(foo(s))]: {add_brackets(add_parentheses(foo(s)))}")
print(f"([foo(s)]): {add_parentheses(add_brackets(foo(s)))}")

foo(s): string is 'hello'
[foo(s)]: [string is 'hello']
(foo(s)): (string is 'hello')
[(foo(s))]: [(string is 'hello')]
([foo(s)]): ([string is 'hello'])


It may seem clever, yet the chain of nested functions can get lengthy and lead to a combinatorial explosion with spaghetti code on the horizon. However, the remedy is on the way. This string of invocations can be reduced and, moreover, (dis/en)abled dynamically with the [Decorator](https://en.wikipedia.org/wiki/Decorator_pattern) pattern.

A warm-up example before jumping to decorators.

In [4]:
def warm(up):
    return up and up


print(warm)  # jus a callable object

f = warm  # that can be referenced by a variable

print(f(8))  # ACKSHUALLY doesn't warm

<function warm at 0x7e773f111c60>
8


In [5]:
def outer(outparam):
    def inner(innparam):
        # outparam is visible here because it is enclosed
        return innparam + outparam
    # outparam is associated with inner even after return
    # see LEGB and closures
    return inner


f = outer(5)  # f = inner
print(f)

f(-5)

<function outer.<locals>.inner at 0x7e773f112c00>


0

Now you are ready to get familiar with decorators. A simple decorator in Python is a callable object (we start from functions) that:

1. receives a callable object -> the function of interest;
2. returns an inner function that reuses the target function and may add new behaviour.

Basic example:

```python
def decorating_function(callable_to_decorate: Callable) -> Callable:
    def wrapper(*args: Any, **kwargs: Any) -> Any:  # so args and kwargs are variadic and can be any -> this is most generic and flexible form
        # some (pre) logic here
        result = callable_to_decorate(*args, **kwargs)
        # some (post) logic here
        # you can return the result if you like

    # you return a callable object itself!  
    return wrapper
```

In [6]:
from collections.abc import Callable

# `bracketize` decorator function
def bracketize(func: Callable) -> Callable:
    # `func` is local within `bracketize` decorator (!)
    # `func` is enclosed (!) for `wrapper`
    def wrapper(*args, **kwargs):
        return f"[{func(*args, **kwargs)}]"
    # return a function that can accept any parameters and produce anything
    # every time `wrapper` is called, `func` is also called (closure)
    return wrapper


# `parenthesize` decorator function
def parenthesize(func: Callable) -> Callable:
    # another decorating function `parenthesize` that suits
    # `_` is a valid name for an actual wrapping function
    # because, again, the decorator pattern is also known as the wrapper pattern
    def _(*args, **kwargs):
        return f"({func(*args, **kwargs)})"
    return _


bra = bracketize(foo)
# bra references to the result of `bracketize`
# which is its inner `wrapper` function
# that aceepts *params, **kwparams
# that are given to the wrapped `foo` function
par = parenthesize(foo)

s1 = "tada"
print(s1)

print(bra(s1))  # case 1 - done
print(par(s1))  # case 2 - done

bra_par = bracketize(par)
# par is callable, so `bracketize` is applicable
# same as `bracketize(parenthesize(foo))`
par_bra = parenthesize(bra)

print(bra_par(s1))
print(par_bra(s1))

tada
[string is 'tada']
(string is 'tada')
[(string is 'tada')]
([string is 'tada'])


In Python, [callable](https://docs.python.org/3/glossary.html#term-callable) objects (these are not only functions) can be decorated with `@decorator` [syntactic sugar](https://en.wikipedia.org/wiki/Syntactic_sugar). It follows Python's idiomaticity when callable objects are meant to be decorated statically, so the decoration becomes permanent once the function is "sugared".

In [7]:
@parenthesize
# since the inner wrapper is `wrapper(*args, **kwargs)`
# the following function can be called without restrictions
@bracketize
# the order matters
def goo(a: int, b: int) -> float:
    return (a + b) ** .5
# equivalent to `goo = parenthesize(bracketize(goo))`

@bracketize
@parenthesize
def hoo(*args) -> int:
    return len(args)
# equivalent to `hoo = bracketize(parenthesize(hoo))`

print(goo(5, 2))
print(hoo(1, 4, 8))

([2.6457513110645907])
[(3)]


Well, I would like to have the result of `goo` rounded to the 2nd digit after the decimal point. Easy, another decorator.

In [8]:
def round_two(cb: Callable) -> Callable:
    def _deco(*args, **kwargs):
        return float(cb(*args, **kwargs))
    return _deco


@parenthesize
@bracketize
@round_two
def goo_v2(a: int, b: int) -> float:
    # not reusing goo on purpose -> consider this standalone
    return round((a + b) ** .5, 2)

# goo_v2 = parenthesize(bracketize(round_two(goo_v2)))


print(goo(5, 2))
print(goo_v2(5, 2))

([2.6457513110645907])
([2.65])


The major benefit about decorators is that we don't need to mess with target functions at all! You just add one decorator or as many as you need.

### Decorators with parameters

The main parameter (and argument) for a decorating function is a callable object that needs to be wrapped. What if passing any other parameters is unavoidable? In the following example, specifying extra parameters does not help.

In [9]:
from typing import Any


# try to remove the default value for `suffix` parameter...aha
def suffux(func: Callable, suffix: str = "") -> Callable:
    def _wrapper(*args, **kwargs):
        return f"{func(*args, **kwargs)} -> {suffix}"
    return _wrapper


def greeter(obj: Any = None) -> str:
    return f"Greetings, {obj}!"


greeter1 = suffux(greeter, "laddie or lassie")
print(greeter1())  # it works, `suffix` parameter is enclosed
print(greeter1("string"))  # prints the expected result


@suffux
def greeter2(obj: Any) -> str:
    return f"Hello, {obj}"


print(greeter2(21))
# and how to change the suffix for the greeter2? :)
# with `greeter1 = suffux(greeter, "laddie or lassie")` it worked
# with greeter2 it is misused!

Greetings, None! -> laddie or lassie
Greetings, string! -> laddie or lassie
Hello, 21 -> 


The situation gets out of hand if the default argument is not provided.

In [10]:
# dammit!
def prefix(func: Callable, prefix: str) -> Callable:
    def wrapper(*args, **kwargs):
        result = func(*args, **kwargs)
        return f"{prefix} -> {result}"
    return wrapper


# Try uncommenting the following cases


# No way anymore
# @prefix
# def answer() -> int:
#     return 42


# Nice try...not at all :)
# @prefix(prefix="no way")
# def answer() -> int:
#     return 42

# The same as `answer = prefix(prefix="no way")`...shit!
# `func` parameter is required!

So what could be done? Here are a few hints:
1. you can call a decorator function with an argument -> `@deco(param)`;
2. this `param` can be an enclosed variable in the scope of the `deco` function;
3. calling `@deco(param)` can return a function that expects a function that wraps ...
4. ...
5. maybe PROFIT!

In [11]:
def deco(prefix: str = "prefix", suffix: str = "suffix") -> Callable:
    def decorator(func: Callable) -> Callable:
        def _wrapper(*args, **kwargs):
            result = func(*args, **kwargs)
            return f"{prefix} > {result} < {suffix}"
        return _wrapper
    return decorator
    # `deco("p", "s")` returns `decorator`
    # that can consume a function,
    # i.e., `decorator(func)` returns a `_wrapper`
    # that is called as `_wrapper(*args, **kwargs)`
    # that invokes the wrapped `func`


@deco("H", "T")
def half_answer() -> int:
    return 21
# equivalent form:
# half_answer = deco("H", "T")(half_answer)

print(half_answer())

H > 21 < T


I guess it looks like PROFIT.

Let's see some more action with a multiplicator decorator.

In [12]:
def multiplier(coef: float = 1) -> Callable:
    def _deco(func: Callable) -> Callable:
        def _(*args, **kwargs):
            return coef * func(*args, **kwargs)
        return _
    return _deco


@multiplier()
def twelve() -> int:
    return 12


@multiplier(-2)
def twenty_one() -> int:
    return 12


@multiplier(.5)
def fourty_two() -> int:
    return 42


print(twelve())  # yes
print(twenty_one())  # not exactly -> -42
print(fourty_two())  # twisted -> 21.0


# even like this
invariant = multiplier(-.5)(multiplier(-2)(lambda: 10))
print(invariant())

12
-24
21.0
10.0


### Data in decorators

There are situations when decorators should have data stored under the hood. If the data is mutable, there are risks that it may change unexpectedly. Consider the example of a decorator that tracks the function call count and also has a small cache for return values to demonstrate the [memoisation](https://en.wikipedia.org/wiki/Memoization) technique.

In [13]:
def memoize(func: Callable) -> Callable:
    def _wrapper(*args, **kwargs):
        # this implementation is not the saviour
        cache: dict[str, Any] = {}
        kws = {k:kwargs[k] for k in sorted(kwargs)}
        if (params := f"{sorted(args)}:{kws}") in cache:
            print(f"{func.__name__}: result from cache")
            return cache[params]
        res = func(*args, **kwargs)
        cache[params] = res
        return res
    return _wrapper


@memoize
def divide(a: int, b: int) -> tuple[int, int]:
    return divmod(a, b)


divide(13, 4)
divide(13, 4)

(3, 1)

This works, but not as expected. Every time the wrapped function is called, the _cache_ local varaible is initialised. When the *_wrapper* scope is left, the cache is gone. Let's move the cache instantiation out of the *_wrapper* function.

In [14]:
def memoize(func: Callable) -> Callable:
    cache: dict[str, Any] = {}
    def _wrapper(*args, **kwargs):
        kws = {k:kwargs[k] for k in sorted(kwargs)}
        if (params := f"{sorted(args)}:{kws}") in cache:
            print(f"{func.__name__}: result from cache")
            return cache[params]
        res = func(*args, **kwargs)
        cache[params] = res
        return res
    return _wrapper


@memoize
def divide(a: int, b: int) -> tuple[int, int]:
    return divmod(a, b)


divide(13, 4)
divide(13, 4)  # from cache

### looks good

@memoize
def answer() -> int:
    return 42

answer()
divide(0, 6)
answer()  # from cache

divide: result from cache
answer: result from cache


42

Now it is better. Every time a function (callable object), shall we say *foo*, is wrapped with the *memoize* decorator, the *cache* is initialised at the time of `foo = memoize(foo)` (or `@memoize` sugar) and for each wrappee the own cache is created. You can guess what may go wrong if a cache were defined globally outside the decorator. The example above can be easily changed to illustrate the point.

### Decorating methods

[Method](https://docs.python.org/3/glossary.html#term-method)s are callable objects associated with a class. The first parameter/argument for a method is either a reference to an instance of this class (*self*) or a class (*cls*). The next example is OK regardless TypeError.

In [15]:
def decoco(func: Callable[[int, int], int]) -> Callable[[int, int], str]:
    def _(a: int, b: int) -> str:
        return f"result = {func(a, b)}"
    return _


@decoco
def add(x: int, y: int) -> int:
    return x + y


print(add(3, 7))  # OK


class Adder:
    @decoco  # signature mismatch
    def add(self, x: int, y: int) -> int:
        return x + y


print(Adder().add(3, 7))

result = 10


TypeError: decoco.<locals>._() takes 2 positional arguments but 3 were given

The decorator requires two parameters, yet the method expects three because of the first self and with this fixed position they can never be in harmony. The decorator should be more flexible because self is unavoidable in methods.

In [16]:
def decoqo(func: Callable[..., int]) -> Callable[..., str]:
    def _(*params, **kwparams) -> str:
        return f"result = {func(*params, **kwparams)}"
    return _


@decoqo
def add(x: int, y: int) -> int:
    return x + y


print(add(3, 7))  # OK


class Adder:
    @decoqo  # whatever signature
    def add(self, x: int, y: int) -> int:
        return x + y

# TypeError: decoco.<locals>._() takes 2 positional arguments but 3 were given
print(Adder().add(3, 7))

result = 10
result = 10


### Classes as decorators

Not only functions can decorate functions/methods. Again, a decorator is a callable object that accepts another callable object. An object is callable if it can be used with parentheses where zero or more (kw)params can be placed, i.e., `obj(*args, kwargs)`. An instance of a class is callable if it defines the [\_\_call\_\_()](https://docs.python.org/3/reference/datamodel.html#object.__call__) dunder. A callable object is tested via the [callable](https://docs.python.org/3/library/functions.html#callable) builtin.

In [17]:
def sumargs(*args, **kwargs):
    return sum([len(args), len(kwargs)])

assert callable(sumargs)
# foo(...) is foo.__call__(...)
assert sumargs(1, 2, a='b') == sumargs.__call__(1, 2, a='b')


class Dummy:
    def __init__(self) -> None:
        self.c = []


assert not callable(Dummy())  # no __call__ dunder


class DummyCallable:
    def __init__(self) -> None:
        self._c: list = []

    def items(self) -> list:
        return self._c[:]

    def __call__(self, *args: Any) -> None:
        self._c.extend(args)


dc = DummyCallable()
assert callable(dc)

dc(1, 2)
dc.__call__(-4, 3)

assert dc.items() == [1, 2, -4, 3]

Time for a class that is callable and designed to decorate other callables.

In [18]:
class Decorator:
    def __init__(self, limit: int = 5) -> None:
        if limit < 0:
            raise ValueError("limit < 0")
        self._limit = limit
        self._func: Callable  # must be defined, e.g. in the __call__

    def _wrapper(self, *args, **kwargs):
        if not self._limit:
            raise ValueError("call limit exceeded")
        res = self._func(*args, **kwargs)
        self._limit -= 1
        return res

    def __call__(self, func: Callable) -> Any:
        # or you can define a wrapper here as an ordinary function
        self._func = func
        return self._wrapper


@Decorator(limit=2)
def power(base: float, exp: float) -> float:
    return base ** exp

# power = Decorator(limit=2)(power)

print(power(2, 4))
print(power(3, 2))
print(power(1, 1))

16
9


ValueError: call limit exceeded

### Decorating classes

We know that functions are definitely callable objects, but what about wrapping classes? Can we decorate them? If so, is there more than one way to do so? Let's just try.

In [19]:
def plog(func: Callable) -> Callable:
    def _(*args: Any, **kwargs: Any) -> Any:
        res =  func(*args, **kwargs)
        print(f"{func.__name__} -> {res}")
        return res
    return _


@plog
class Dummy:
    def __init__(self, number: float) -> None:
        self._nbr = number

    def __repr__(self) -> str:
        return f"{type(self).__name__}({self._nbr})"

    @property
    def number(self) -> float:
        return self._nbr

    def scale(self, coef: float) -> float:
        return self.number * coef

dummy = Dummy(5)  # only here the decorator works
print()
print(f"scaled by -2: {dummy.scale(-2)}")  # no effect
print(f"number = {dummy.number}")  # no plogging is done

Dummy -> Dummy(5)

scaled by -2: -10
number = 5


So, the decorating logic works only when the class is instantiated and does not affect other methods.

In fact, the classes are callable. From the [docs](https://docs.python.org/3/reference/datamodel.html#classes):
> Classes are callable. These objects normally act as factories for new instances of themselves, but variations are possible for class types that override [\_\_new\_\_()](https://docs.python.org/3/reference/datamodel.html#object.__new__). The arguments of the call are passed to \_\_new\_\_() and, in the typical case, to [\_\_init\_\_()](https://docs.python.org/3/reference/datamodel.html#object.__init__) to initialize the new instance.

A [quote](https://docs.python.org/3/reference/datamodel.html#class-instances) about class instances:
> Instances of arbitrary classes can be made callable by defining a \_\_call\_\_() method in their class.

So, wrapping a class is like wrapping its \_\_init\_\_() dunder. It makes sense because calling a class means creating and returning its instance.

### Wrapping coroutines

Coroutines are easy to wrap yet there are some points to remember:
- `await` inside a sync `def` function is SyntaxError
- so, a wrapper should be an `async def` function

In [20]:
def wrap_coro(coro: Callable) -> Callable:
    async def _wrapper(*args, **kwargs):
        print(f"before {coro}")
        result = await coro(*args, **kwargs)
        print(f"after {coro} -> {result}")
        return result
    return _wrapper


@wrap_coro
async def coro(*args):
    print(f"I am just coroutine with {args}")


await coro()

before <function coro at 0x7e773db6cea0>
I am just coroutine with ()
after <function coro at 0x7e773db6cea0> -> None


Can we have a single decorator for wrapping either a routine or a coroutine? It is doable and [this solution](https://stackoverflow.com/questions/44169998/how-to-create-a-python-decorator-that-can-wrap-either-coroutine-or-function) looks good to me yet it may be a step to overengineering.

In [21]:
import inspect
from contextlib import contextmanager


def powerful_decorator(func):
    @contextmanager
    def wrapping_logic():
        print("Hello")
        yield
        print("Bye")

    def wrapper(*args, **kwargs):
        if not inspect.iscoroutinefunction(func):
            with wrapping_logic():
                return func(*args, **kwargs)
        print("Wrapping a coroutine")
        async def tmp():
            with wrapping_logic():
                return (await func(*args, **kwargs))
        return tmp()

    return wrapper


@powerful_decorator
def synco():
    print("Time to sync")


@powerful_decorator
async def asynco():
    print("Our sync is run async")


synco()
await asynco()

Hello
Time to sync
Bye
Wrapping a coroutine
Hello
Our sync is run async
Bye


### Docstrings of decorated functions

So far so good. We can decorate functions, methods, classes without boring boilerplate. But how do decorators work with documented entities?

In [22]:
def docstring_perdu(cb: Callable) -> Callable:
    def _wrapper(*args, **kwargs):
        return cb(*args, **kwargs)
    return _wrapper


def divide_without_conquering(a: float, b: float) -> float:
    """Perform `a / b` logic.

    Args:
        a (float): dividend
        b (float): divisor

    Raises:
        ZeroDivisionError

    Returns:
        int
    """
    return a / b


print(divide_without_conquering.__doc__)  # OK

decorated = docstring_perdu(divide_without_conquering)
print(decorated.__doc__)  # What ?!

Perform `a / b` logic.

    Args:
        a (float): dividend
        b (float): divisor

    Raises:
        ZeroDivisionError

    Returns:
        int
    
None


The docstring is lost and not found because the *_wrapper* does not have it. The idea of specifying a docstring in the *_wrapper* is of no use:

- the wrapped object must not be stripped of its docstring, obviously;
- docstring in the *_wrapper* does not respect DRY (Don't Repeat Yourself) principle;
- what will happen if the same decorator wraps different functions with different docstring?

Fortunately, there is a much more fancy solution -> [functools.wraps](https://docs.python.org/3/library/functools.html).


In [23]:
import functools

def docstring_saved(cb: Callable) -> Callable:
    @functools.wraps(cb)  # accepts the target cb and wraps the wrapper
    def _wrapper(*args, **kwargs):
        return cb(*args, **kwargs)
    return _wrapper


@docstring_saved
def divide_without_conquering(a: float, b: float) -> float:
    """Perform `a / b` logic.

    Args:
        a (float): dividend
        b (float): divisor

    Raises:
        ZeroDivisionError

    Returns:
        int
    """
    return a / b


print(divide_without_conquering.__doc__)  # OK

Perform `a / b` logic.

    Args:
        a (float): dividend
        b (float): divisor

    Raises:
        ZeroDivisionError

    Returns:
        int
    


Nicely done.

### Wrapping up

Let's implement a simple [Retry](https://www.geeksforgeeks.org/system-design/retry-pattern-in-microservices/) strategy to handle temporary/occasional failures related to I/O operations. Imagine there is an async `get_data` (coroutine) function that requests data from a remote source. If a request fails because of some connection issues, the function is reinvoked until it reaches certain limits.

We understand that:
- the number of retries is supposed to be limited;
- only certain exceptions should be intercepted and not in a blind fashion;
- the delay between attempts is desirable to prevent overwhelming the system with \[D\]DoS retries.

The `retry` decorator will handle the case, yet it will be kept as simple as possible and avoid the [Retry Storm situation](https://dev.to/willvelida/the-retry-pattern-and-retry-storm-anti-pattern-4k6k).

In [24]:
import asyncio
import functools
import random

from collections.abc import Iterable


_CODES = [200, 201, 400, 500, 502]


def _respond() -> int:
    if random.randint(0, 2) == 2:
        raise Exception("unexpected")
    if (code := random.choice(_CODES)) == 500:
        raise ConnectionError("I'm tired")
    print(code)
    return code


async def get_data(url: str) -> dict[str, Any]:
    # simulate I/O
    await asyncio.sleep(random.random())
    return {f"{url}": _respond()}


def retry(times: int = 3, *, backoff: float = 1, exceptions: Iterable[Exception] = (Exception,)) -> Callable:
    def _deco(func: Callable) -> Callable:
        nonlocal exc
        @functools.wraps(func)
        async def _wrapper(*args: Any, **kwargs: Any) -> Any:
            fdm = f"{func.__name__}(args={args}, kwargs={kwargs})"
            for attempt in range(1, times + 1):
                try:
                    print(fdm)
                    return await func(*args, **kwargs)
                except exc as e:
                    print(f"request failed for {fdm}: {e}")
                    if attempt == times:
                        print("No chance")
                        raise
                    jitter = random.random()
                    # here the backoff strategy is simple (no exponential rate)
                    # yet with jitter -> random addendum between attempts
                    await asyncio.sleep(max(1, backoff + jitter))
        return _wrapper
    exc = tuple(exceptions)
    return _deco


async def main() -> None:
    f = retry(exceptions=[ConnectionError])(get_data)
    tasks: list[asyncio.Task] = []
    for i in range(1, 11):
        tasks.append(asyncio.create_task(f(f"URL-{i}")))
    results = await asyncio.gather(*tasks, return_exceptions=True)
    print(results)
    # not interested in results here


await main()

get_data(args=('URL-1',), kwargs={})
get_data(args=('URL-2',), kwargs={})
get_data(args=('URL-3',), kwargs={})
get_data(args=('URL-4',), kwargs={})
get_data(args=('URL-5',), kwargs={})
get_data(args=('URL-6',), kwargs={})
get_data(args=('URL-7',), kwargs={})
get_data(args=('URL-8',), kwargs={})
get_data(args=('URL-9',), kwargs={})
get_data(args=('URL-10',), kwargs={})
200
request failed for get_data(args=('URL-7',), kwargs={}): I'm tired
400
400
400
201
200
201
request failed for get_data(args=('URL-8',), kwargs={}): I'm tired
get_data(args=('URL-7',), kwargs={})
get_data(args=('URL-8',), kwargs={})
400
200
[Exception('unexpected'), {'URL-2': 400}, {'URL-3': 400}, {'URL-4': 400}, {'URL-5': 201}, {'URL-6': 200}, {'URL-7': 200}, {'URL-8': 400}, {'URL-9': 200}, {'URL-10': 201}]


Easy peasy lemon squeezy.

### Summary

Honestly, I thought that [Decorator](https://en.wikipedia.org/wiki/Decorator_pattern), also referred to as Wrapper, is a *behavioural* design pattern since it allows us to change the behaviour of a decorated callable object. Yet this pattern is classified as **structural** and such patterns focus on the composition and (re)structuring, hence the category. The fact that Decorator/Wrapper is structural has solid reasons:

1. A decorated callable changes neither in structure nor behaviour.
2. A wrapper over it uses the wrappee as a component which means composition (preferred to inheritance).
3. The resultant object expresses the relationship between the decorator and the wrappee that results to a new modified structure.

Suppose there is a function A that relies on two helper functions B and C. In the body of A function B comes first and then C. If you swap them, you change the structure (!) of the function A and by this its behaviour (logic) is changed! This explanation may be enough (at least for me) that "logic" and "behaviour" should not used interchangeably unless the context blesses to do so.

That is all for now.